# Merge Old Surrey Districts into East Surrey & West Surrey

Takes the LAD GeoJSON and:
1. Merges 5 districts into **East Surrey** (Elmbridge, Epsom and Ewell, Mole Valley, Reigate and Banstead, Tandridge)
2. Merges 6 districts into **West Surrey** (Guildford, Runnymede, Spelthorne, Surrey Heath, Waverley, Woking)
3. Removes the 11 old district features
4. Adds 2 new unitary authority features
5. Downloads the modified GeoJSON

**Upload `lad_map.geojson` when prompted.**

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import json
from shapely.geometry import shape, mapping
from shapely.ops import unary_union

# Load the GeoJSON
with open('lad_map.geojson') as f:
    geojson = json.load(f)

print(f"Loaded {len(geojson['features'])} features")

In [ ]:
# Define the two new unitary authorities
EAST_SURREY = ['Elmbridge', 'Epsom and Ewell', 'Mole Valley', 'Reigate and Banstead', 'Tandridge']
WEST_SURREY = ['Guildford', 'Runnymede', 'Spelthorne', 'Surrey Heath', 'Waverley', 'Woking']
ALL_OLD = set(EAST_SURREY + WEST_SURREY)

# Separate Surrey districts from everything else
east_geoms = []
west_geoms = []
other_features = []

for feat in geojson['features']:
    name = feat['properties']['LAD25NM']
    if name in EAST_SURREY:
        east_geoms.append(shape(feat['geometry']))
        print(f"  East Surrey <- {name} ({feat['properties']['LAD25CD']})")
    elif name in WEST_SURREY:
        west_geoms.append(shape(feat['geometry']))
        print(f"  West Surrey <- {name} ({feat['properties']['LAD25CD']})")
    else:
        other_features.append(feat)

print(f"\nFound {len(east_geoms)} East Surrey districts, {len(west_geoms)} West Surrey districts")
print(f"Remaining features: {len(other_features)}")

assert len(east_geoms) == 5, f"Expected 5 East Surrey districts, got {len(east_geoms)}"
assert len(west_geoms) == 6, f"Expected 6 West Surrey districts, got {len(west_geoms)}"

In [ ]:
# Merge geometries
east_merged = unary_union(east_geoms)
west_merged = unary_union(west_geoms)

# Compute centroids for BNG_E/N and LONG/LAT
east_centroid = east_merged.centroid
west_centroid = west_merged.centroid

# Build new features matching the LAD25 property schema
# Using placeholder codes — no official ONS codes exist yet
east_feature = {
    'type': 'Feature',
    'properties': {
        'FID': max(f['properties']['FID'] for f in other_features) + 1,
        'LAD25CD': 'E06000083',  # Placeholder — adjust if ONS assigns different code
        'LAD25NM': 'East Surrey',
        'LAD25NMW': ' ',
        'BNG_E': None,
        'BNG_N': None,
        'LONG': round(east_centroid.x, 8),
        'LAT': round(east_centroid.y, 8),
        'GlobalID': ''
    },
    'geometry': mapping(east_merged)
}

west_feature = {
    'type': 'Feature',
    'properties': {
        'FID': max(f['properties']['FID'] for f in other_features) + 2,
        'LAD25CD': 'E06000084',  # Placeholder — adjust if ONS assigns different code
        'LAD25NM': 'West Surrey',
        'LAD25NMW': ' ',
        'BNG_E': None,
        'BNG_N': None,
        'LONG': round(west_centroid.x, 8),
        'LAT': round(west_centroid.y, 8),
        'GlobalID': ''
    },
    'geometry': mapping(west_merged)
}

print(f"East Surrey: {east_merged.geom_type}, area={east_merged.area:.6f}")
print(f"West Surrey: {west_merged.geom_type}, area={west_merged.area:.6f}")

In [ ]:
# Build final GeoJSON
output = {
    'type': 'FeatureCollection',
    'name': geojson.get('name', 'LAD_boundaries'),
    'crs': geojson.get('crs'),
    'features': other_features + [east_feature, west_feature]
}

# Remove crs key if None
if output['crs'] is None:
    del output['crs']

outfile = 'lad_map.geojson'
with open(outfile, 'w') as f:
    json.dump(output, f)

print(f"Written {len(output['features'])} features to {outfile}")
print(f"  (was {len(geojson['features'])}, removed 11, added 2 = net -9)")

In [ ]:
# Quick sanity check
surrey_names = [f['properties']['LAD25NM'] for f in output['features'] 
                if 'surrey' in f['properties']['LAD25NM'].lower()]
print(f"Surrey features in output: {surrey_names}")

old_remaining = [f['properties']['LAD25NM'] for f in output['features'] 
                 if f['properties']['LAD25NM'] in ALL_OLD]
print(f"Old districts still present: {old_remaining if old_remaining else 'None ✓'}")

In [ ]:
# Download the modified file
files.download(outfile)